In [1]:
import os
import pandas as pd


In [2]:
%pwd

'c:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps\\research'

In [3]:
os.chdir(r"C:\Users\sam\End-to-End-Fashion-Recommendation-System-with-MLOps")


In [4]:
%pwd

'C:\\Users\\sam\\End-to-End-Fashion-Recommendation-System-with-MLOps'

In [5]:
import numpy as np

In [15]:
from dataclasses import dataclass
from pathlib import Path
@dataclass

class model_evalulation_path:
    sim_matrix_path: Path
    featured_df_path :Path 
    model_eval_metrics : Path
    ml_flow_tracking_uri :str
    ml_flow_experiment_name :str
    ml_flow_run_name :str



In [7]:
from src.recommendation_system.constants import CONFIG_PATH
from src.recommendation_system.utils.common import read_yaml , create_dir
from dataclasses import dataclass
from src.recommendation_system.logging import logger
import pickle

In [16]:
class Congif_manager:

    def __init__(self, config =CONFIG_PATH):
        self.config = read_yaml(config)

        create_dir([self.config.artifacts_root])

    def get_model_evaluation(self) -> model_evalulation_path:
        config = self.config.model_evalvation

        create_dir([config.model_eval_metrics])

        return model_evalulation_path(
            sim_matrix_path        = (config.sim_matrix_path),
            featured_df_path       = (config.featured_df_path),
            model_eval_metrics = (config.model_eval_metrics),
            ml_flow_tracking_uri   = config.ml_flow_tracking_uri,
        ml_flow_experiment_name = config.ml_flow_experiment_name,
        ml_flow_run_name        = config.ml_flow_run_name
        ) 

In [ ]:
class model_evalulation_congif:

    def __init__(self, config):
        self.config     = config
        self.sim_matrix = None
        self.df         = None

    def load_artifacts(self):
        try:
            logger.info("=" * 20 + " Loading Artifacts STARTED " + "=" * 20)
            with open(self.config.sim_matrix_path, 'rb') as f:
                self.sim_matrix = pickle.load(f)
            with open(self.config.featured_df_path, 'rb') as f:
                self.df = pickle.load(f)
            logger.info("sim_matrix : %s", str(self.sim_matrix.shape))
            logger.info("df         : %s", str(self.df.shape))
            logger.info("=" * 20 + " Loading Artifacts COMPLETED " + "=" * 20)

        except Exception as e:
            logger.error("Load artifacts FAILED - %s", str(e))
            raise e

    def get_test_sample(self) -> pd.DataFrame:
        try:
            logger.info("=" * 20 + " Test Sample STARTED " + "=" * 20)

            # 5 products per aesthetic — balanced test set
            test_set = (
                self.df.groupby("aesthetic", group_keys=False)
                       .apply(lambda x: x.sample(min(5, len(x)), random_state=42))
                       .drop_duplicates()
                       .reset_index(drop=True)
            )

            logger.info("Sample size : %d", len(test_set))
            logger.info("=" * 20 + " Test Sample COMPLETED " + "=" * 20)
            return test_set

        except Exception as e:
            logger.error("Test sample FAILED - %s", str(e))
            raise e

    # ── ground truth — similarity threshold ──────────
    def get_relevant(self, product_id, threshold=0.4):
        scores = self.sim_matrix[product_id].copy()
        return [
            i for i, s in enumerate(scores)
            if s >= threshold and i != product_id
        ]

    # ── precision@k ──────────────────────────────────
    def precision_at_k(self, top_idx, relevant_idx, k) -> float:
        hits = len(set(top_idx[:k]) & set(relevant_idx))
        return hits / k if k > 0 else 0.0

    # ── ndcg@k ───────────────────────────────────────
    def ndcg_at_k(self, top_idx, relevant_idx, k) -> float:
        relevant_set = set(relevant_idx)
        dcg  = sum(
            1 / np.log2(i + 2)
            for i, idx in enumerate(top_idx[:k])
            if idx in relevant_set
        )
        idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(relevant_set))))
        return dcg / idcg if idcg > 0 else 0.0

    # ── map ──────────────────────────────────────────
    def map_score(self, top_idx, relevant_idx, k) -> float:
        relevant_set = set(relevant_idx)
        hits, score  = 0, 0.0
        for i, idx in enumerate(top_idx[:k]):
            if idx in relevant_set:
                hits  += 1
                score += hits / (i + 1)
        return score / len(relevant_set) if relevant_set else 0.0

    def score_product(self, row, top_k=10, threshold=0.5) -> dict:
        try:
            product_id   = row.name
            relevant_idx = self.get_relevant(product_id, threshold)

            scores             = self.sim_matrix[product_id].copy()
            scores[product_id] = -1
            top_idx            = scores.argsort()[::-1][0:top_k]

            precision = self.precision_at_k(top_idx, relevant_idx, top_k)
            ndcg      = self.ndcg_at_k(top_idx,      relevant_idx, top_k)
            map_s     = self.map_score(top_idx,       relevant_idx, top_k)

            logger.info(
                "%-35s | P@K=%.2f | NDCG=%.2f | MAP=%.2f",
                row["product_name_clean"][:35],
                precision, ndcg, map_s
            )

            return {
                "product_name_clean" : row["product_name_clean"],
                "aesthetic"          : row["aesthetic"],
                "n_relevant"         : len(relevant_idx),
                "precision_at_k"     : round(precision, 4),
                "ndcg_at_k"          : round(ndcg,      4),
                "map"                : round(map_s,     4),
            }

        except Exception as e:
            logger.error("score_product FAILED - %s", str(e))
            raise e

    def run_evaluation(self, top_k=10, threshold=0.4):
        try:
            logger.info("=" * 20 + " Evaluation STARTED " + "=" * 20)

            if self.df is None or self.sim_matrix is None:
                self.load_artifacts()

            test_sample = self.get_test_sample()
            logger.info("Scoring %d products", len(test_sample))

            records = []
            for _, row in test_sample.iterrows():
                result = self.score_product(row, top_k=top_k, threshold=threshold)
                records.append(result)

            metrics_df = pd.DataFrame(records)

            summary = {
                "n_products_tested" : len(metrics_df),
                "top_k"             : top_k,
                "threshold"         : threshold,
                "precision_at_k"    : round(metrics_df["precision_at_k"].mean(), 4),
                "ndcg_at_k"         : round(metrics_df["ndcg_at_k"].mean(),      4),
                "map"               : round(metrics_df["map"].mean(),             4),
            }

            logger.info("======= SUMMARY =======")
            for k, v in summary.items():
                logger.info("%-25s : %s", k, v)

            # save
            out_path = os.path.join(
                self.config.model_eval_metrics, 'eval_metrics.csv'
            )
            metrics_df.to_csv(out_path, index=False)
            logger.info("Saved -> %s", out_path)
            logger.info("=" * 20 + " Evaluation COMPLETED " + "=" * 20)

            
        
            # ── MLflow ───────────────────────────────
            mlflow.set_tracking_uri(self.config.ml_flow_tracking_uri)
            mlflow.set_experiment(self.config.ml_flow_experiment_name)

            with mlflow.start_run(run_name=self.config.ml_flow_run_name) as run:

                mlflow.log_param("top_k",     top_k)
                mlflow.log_param("threshold", threshold)
                mlflow.log_param("n_tested",  len(metrics_df))

                mlflow.log_metric("precision_at_k", summary["precision_at_k"])
                mlflow.log_metric("ndcg_at_k",      summary["ndcg_at_k"])
                mlflow.log_metric("map",            summary["map"])

                mlflow.log_artifact(out_path)

            run_id        = run.info.run_id
            experiment_id = run.info.experiment_id
            run_url       = f"{self.config.ml_flow_tracking_uri}/#/experiments/{experiment_id}/runs/{run_id}"

    logger.info("Run ID  : %s", run_id)
    logger.info("Run URL : %s", run_url)
        return summary, metrics_df
    except Exception as e:
        logger.error("Evaluation FAILED - %s", str(e))
        raise e

In [14]:
con   = Congif_manager()
model = con.get_model_evaluation()
model = model_evalulation_congif(model)
model.run_evaluation()   



[2026-03-05 12:01:19,575: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-05 12:01:19,579: INFO: common: created directory at: artifacts]
[2026-03-05 12:01:19,581: INFO: common: created directory at: artifacts/model_evalvation/model/]
[2026-03-05 12:01:19,807: INFO: 3091269413: ==================== Evaluation STARTED ====================]
[2026-03-05 12:01:19,815: INFO: 3091269413: ==================== Loading Artifacts STARTED ====================]
[2026-03-05 12:01:23,525: INFO: 3091269413: sim_matrix : (11951, 11951)]
[2026-03-05 12:01:23,527: INFO: 3091269413: df         : (11951, 16)]
[2026-03-05 12:01:23,529: INFO: 3091269413: ==================== Loading Artifacts COMPLETED ====================]
[2026-03-05 12:01:23,530: INFO: 3091269413: ==================== Test Sample STARTED ====================]
[2026-03-05 12:01:23,555: INFO: 3091269413: Sample size : 35]
[2026-03-05 12:01:23,557: INFO: 3091269413: ==================== Test Sample COMPLETED ======

C:\Users\sam\AppData\Local\Temp\ipykernel_22300\3091269413.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(5, len(x)), random_state=42))


[2026-03-05 12:01:23,729: INFO: 3091269413: mens solid burgundy baggy fit cargo | P@K=0.20 | NDCG=1.00 | MAP=1.00]
[2026-03-05 12:01:23,734: INFO: 3091269413: styleheaven mens fashion button dow | P@K=0.40 | NDCG=1.00 | MAP=1.00]
[2026-03-05 12:01:23,737: INFO: 3091269413: men classic polo tshirt short sleev | P@K=0.30 | NDCG=1.00 | MAP=1.00]
[2026-03-05 12:01:23,745: INFO: 3091269413: men casual spread collar shirt half | P@K=0.00 | NDCG=0.00 | MAP=0.00]
[2026-03-05 12:01:23,750: INFO: 3091269413: women cotton ethnic co ord set kurt | P@K=0.20 | NDCG=1.00 | MAP=1.00]
[2026-03-05 12:01:23,759: INFO: 3091269413: men polyester regular jacket        | P@K=0.10 | NDCG=1.00 | MAP=1.00]
[2026-03-05 12:01:23,772: INFO: 3091269413: ======= SUMMARY =======]
[2026-03-05 12:01:23,775: INFO: 3091269413: n_products_tested         : 35]
[2026-03-05 12:01:23,777: INFO: 3091269413: top_k                     : 10]
[2026-03-05 12:01:23,783: INFO: 3091269413: threshold                 : 0.4]
[2026-03-05 

({'n_products_tested': 35,
  'top_k': 10,
  'threshold': 0.4,
  'precision_at_k': np.float64(0.3857),
  'ndcg_at_k': np.float64(0.9143),
  'map': np.float64(0.8946)},
                                    product_name_clean       aesthetic  \
 0   khaki button front closure classic flap pocket...  college_casual   
 1                    men slim fit mid rise track pant  college_casual   
 2   men mid rise premium skinny fit stretchable jeans  college_casual   
 3   men skypee casual sneaker lightweight lace up ...  college_casual   
 4   men regular fit solid pattern polyester blend ...  college_casual   
 5   men premium waffle knit oversized tshirt textu...      date_clean   
 6                   men blazer casual and formal wear      date_clean   
 7   men checks roll up sleeves slim fit linen blen...      date_clean   
 8   combo of men s premium satin cotton shirt with...      date_clean   
 9   men relaxed fit shirts printed half sleeves cu...      date_clean   
 10  fleece regular

In [29]:
import pandas as pd 
df = pd.read_csv(r'artifacts\data_transformation\transformed data\Transformed_Data.csv')

In [15]:
import pickle

with open(r'artifacts\feature_engineering\model\sim_matrix.pkl', 'rb') as f:
    sim_matrix = pickle.load(f)
    sim_matrix

In [20]:
sim_matrix[1]

array([0.15691217, 1.        , 0.09687231, ..., 0.        , 0.        ,
       0.        ], shape=(13813,))

In [27]:
sim_matrix[54].argsort()[::-1][0:10]

array([  54, 2114, 8453, 2172,  106, 8440,  163, 8580,  258, 8539])

In [73]:
df[(df["aesthetic"] == 'streetwear')].index[1::][1:10]

Index([2, 3, 4, 5, 6, 7, 8, 9, 10], dtype='int64')

In [72]:
sim_matrix[1].argsort()[::-1][1:10]

array([8421, 2095,    1,   43, 8428,    5, 8426, 7354, 8435])

In [65]:
relevant_idx = df[
    (df["aesthetic"] == aesthetic) &
    (df.index != product_id)
].index.tolist()

In [66]:
print("relevant_idx count :", len(relevant_idx))
print("first 10           :", relevant_idx[:10])

relevant_idx count : 2581
first 10           : [3946, 3947, 3948, 3949, 3950, 3951, 3952, 3953, 3954, 3955]


In [59]:
df.loc[2000]

asin                                                     B0D1XXW98D
product_name      Unisex Regular Cotton Combo of 3 Solid Color T...
price                                                      6.809039
rating                                                          4.2
review_count                                               6.888572
discount                                                         55
image_url         https://m.media-amazon.com/images/I/41ITY79hWU...
product_link      https://www.amazon.in/XENOVAURBAN-Unisex-Regul...
aesthetic                                            college_casual
is_new_product                                                    0
Name: 2000, dtype: object